### PyTorch

The framework used throughout this series (`nn.RNN`/`nn.LSTM` in `nlp.ipynb`, `nn.Dropout` in `regularization.ipynb`, `nn.MultiheadAttention`-style computation in `llm-mechanics/llm-architecture.ipynb`). Covers tensors, autograd, nn.Module, the training loop, tying back to the hand-derived gradients from earlier notebooks to confirm autograd computes the exact same thing automatically.

#### 0. Tensors

Like numpy arrays, with two things numpy does not have: GPU support (`.to("cuda")` moves computation onto a GPU with no code change beyond that), and autograd (tensors can track the operations applied to them, enabling automatic gradient computation, covered next).

#### 0.5. Weight initialization

Why not start at zero: if every weight in a layer starts identical (e.g. all zero), every neuron computes the identical output and the identical gradient, they stay identical forever, a symmetry problem, the layer effectively has only 1 unique neuron's worth of learning capacity no matter how many neurons it has.

Why not just large random values: large weights push activations into saturated regions of sigmoid/tanh (see `activation-functions.ipynb`'s worked gradient-shrinkage example), or cause activations to explode layer over layer in a deep network.

Xavier/Glorot initialization (designed for sigmoid/tanh): draw weights so the VARIANCE stays roughly constant from layer to layer, scaled by fan_in and fan_out (the number of inputs and outputs of that layer). Uniform version: limit = sqrt(6/(fan_in+fan_out)), draw uniformly from [-limit, limit].

Worked example, a layer with fan_in=100, fan_out=50:
```
Xavier limit = sqrt(6/(100+50)) = sqrt(0.04) = 0.2
weights drawn uniformly from [-0.2, 0.2]
```
He initialization (designed for ReLU): variance = 2/fan_in, accounts for ReLU zeroing out roughly half of all activations on average (see `activation-functions.ipynb`'s dead ReLU section), so it compensates with a larger variance than Xavier would give, to keep the SURVIVING half of activations at a healthy scale.
```
He variance = 2/100 = 0.02, std = sqrt(0.02) = 0.141
weights drawn from Normal(0, 0.141^2)
```
Rule of thumb: Xavier for sigmoid/tanh layers, He for ReLU layers, PyTorch's `nn.Linear` uses a variant of He-style initialization by default already, this is rarely something you set manually, but knowing why it is not just zeros or arbitrary randomness matters for debugging a network that will not train.

In [ ]:
import torch
import torch.nn as nn

layer = nn.Linear(100, 50)

xavier_layer = nn.Linear(100, 50)
nn.init.xavier_uniform_(xavier_layer.weight)

he_layer = nn.Linear(100, 50)
nn.init.kaiming_normal_(he_layer.weight, nonlinearity="relu")  # "kaiming" is He's other name

print("default init weight std:", layer.weight.std().item())
print("Xavier init weight range: [", xavier_layer.weight.min().item(), ",", xavier_layer.weight.max().item(), "] (expect near +-0.2)")
print("He init weight std:", he_layer.weight.std().item(), "(expect near 0.141)")

In [ ]:
import torch

x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])

print("element-wise add:", x + y)
print("dot product:", torch.dot(x, y))
print("shape:", x.shape, "| dtype:", x.dtype)
print("device:", x.device, "| cuda available:", torch.cuda.is_available())

#### 1. Autograd, verified against a hand-derived gradient

This is the actual mechanism behind every gradient formula worked out by hand across this series (logreg's `(p-y)*x` in `classical-ml.ipynb`, XGBoost's `g=p-y` in `boosting.ipynb`), PyTorch computes them automatically via the chain rule applied to whatever operations were run on tensors with `requires_grad=True`.

Worked check: same single-point logistic regression setup as `classical-ml.ipynb`'s from-scratch gradient derivation. x=2.0, w=0.5, b=0.1, true label y=1.
```
z = w*x + b = 0.5*2 + 0.1 = 1.1
p = sigmoid(1.1) = 0.7503
loss (BCE) = -log(p) = 0.2872   (y=1, so only the log(p) term survives)

hand-derived gradient: dL/dw = (p-y)*x = (0.7503-1)*2 = -0.4994
                        dL/db = (p-y)*1 = -0.2497
```
If autograd is doing the same chain-rule computation the derivation did by hand, `w.grad` and `b.grad` after `.backward()` should come out to those exact numbers.

In [ ]:
import torch

x = torch.tensor(2.0)
y = torch.tensor(1.0)
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)

z = w * x + b
p = torch.sigmoid(z)
loss = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))

loss.backward()

print("p:", p.item(), "| loss:", loss.item())
print("autograd dL/dw:", w.grad.item(), "(hand-derived: -0.4994)")
print("autograd dL/db:", b.grad.item(), "(hand-derived: -0.2497)")

#### 2. nn.Module: the same logistic regression, as a proper model

Wrapping weights and the forward computation into a class, sklearn's/this series' `LogisticRegression()`, `RandomForestClassifier()` etc. all follow a similar fit/predict pattern, `nn.Module` is PyTorch's version, you define `forward()` (what to compute) and the framework handles `backward()` (how to differentiate it) automatically via autograd.

In [ ]:
import torch.nn as nn

class LogisticRegressionModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)  # holds w and b internally

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

model = LogisticRegressionModel(n_features=2)
sample_input = torch.tensor([[1.0, 2.0]])
print("model output (untrained, random weights):", model(sample_input))
print("parameters:", list(model.parameters()))

#### 3. The training loop

Canonical 5-step loop, the same conceptual loop as every `for round in range(n_estimators)` boosting loop and every from-scratch `for i in range(n_iters)` gradient descent loop in this series, just using autograd instead of a manually derived gradient formula:
```
for epoch in range(n_epochs):
    optimizer.zero_grad()      # clear gradients from the previous step, they accumulate otherwise
    predictions = model(X)     # forward pass
    loss = loss_fn(predictions, y)
    loss.backward()            # autograd computes every gradient
    optimizer.step()           # w -= lr * w.grad, for every parameter, done automatically
```
`optimizer.zero_grad()` matters specifically because PyTorch ACCUMULATES gradients into `.grad` by default (adds to whatever was already there, does not overwrite), useful for some advanced patterns (gradient accumulation across mini-batches too large to fit in memory at once), but means a normal loop that forgets this step silently sums gradients across every epoch, producing wrong, ever-growing updates.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
X = torch.randn(100, 2)
y = (X[:, 0] + X[:, 1] > 0).float().unsqueeze(1)

model = LogisticRegressionModel(n_features=2)
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

losses = []
for epoch in range(200):
    optimizer.zero_grad()
    preds = model(X)
    loss = loss_fn(preds, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print("loss at epoch 0:", losses[0])
print("loss at epoch 199:", losses[-1])
print("learned weights:", model.linear.weight.detach())

#### 4. Optimizers, worked by hand

SGD: plain gradient descent, `w -= lr * grad`, exactly the update rule derived by hand throughout this series. Reacts to only the CURRENT gradient, no memory of past steps, so it oscillates on noisy gradients.

Momentum: accumulate a running average of past gradients, step using that smoothed direction instead of the raw noisy one. Formula: v = beta*v_prev + grad, then w -= lr*v (beta commonly 0.9).

Worked example, a noisy, oscillating gradient sequence over 3 steps: [1.0, -0.8, 1.2]
```
v1 = 0.9*0 + 1.0 = 1.0
v2 = 0.9*1.0 + (-0.8) = 0.9 - 0.8 = 0.1
v3 = 0.9*0.1 + 1.2 = 0.09 + 1.2 = 1.29
```
Compare the raw gradients [1.0, -0.8, 1.2] (flips sign every step) against the momentum-smoothed v sequence [1.0, 0.1, 1.29] (stays positive throughout, the accumulated positive momentum from step 1 damps step 2's negative gradient rather than fully reversing direction). This is the actual mechanism behind "momentum smooths out noisy updates."

RMSprop: adaptive PER-PARAMETER learning rate, divide the step by a running average of squared gradients, so parameters with consistently large gradients get automatically smaller effective steps (preventing overshoot) and parameters with small gradients get relatively larger steps (preventing them from stalling). Formula: s = beta*s_prev + (1-beta)*grad^2, w -= lr*grad/sqrt(s+eps).

Adam: combines BOTH ideas, momentum (a running average of the gradient itself, called the first moment) and RMSprop (a running average of the squared gradient, the second moment), plus a bias correction for the first few steps (since both running averages start at 0 and are biased low early on). The default choice for training neural networks in practice.

AdamW: Adam plus DECOUPLED weight decay, applies the L2 regularization penalty directly to the weight update, separately from the gradient-based adaptive scaling, rather than folding it into the gradient the way plain Adam does. Fixes a subtle bug-like interaction where Adam's per-parameter adaptive scaling distorts what L2 regularization is supposed to do (shrink every weight by the same proportional amount, see `regularization.ipynb`'s Ridge section, Adam's adaptive denominator breaks that uniformity if weight decay is folded into the gradient instead of applied separately).


In [ ]:
gradients = [1.0, -0.8, 1.2]
beta = 0.9
v = 0.0
for g in gradients:
    v = beta * v + g
    print(f"gradient={g}, momentum v={v:.3f}")

# compare optimizers on the same toy problem
import torch
import torch.nn as nn

torch.manual_seed(0)
X = torch.randn(50, 2)
y = (X[:, 0] + X[:, 1] > 0).float().unsqueeze(1)

for name, opt_class, kwargs in [
    ("SGD", torch.optim.SGD, {"lr": 0.1}),
    ("SGD+momentum", torch.optim.SGD, {"lr": 0.1, "momentum": 0.9}),
    ("RMSprop", torch.optim.RMSprop, {"lr": 0.01}),
    ("Adam", torch.optim.Adam, {"lr": 0.01}),
    ("AdamW", torch.optim.AdamW, {"lr": 0.01, "weight_decay": 0.01}),
]:
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
    optimizer = opt_class(model.parameters(), **kwargs)
    loss_fn = nn.BCELoss()
    for _ in range(50):
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
    print(f"{name}: loss after 50 steps = {loss.item():.4f}")

#### 5. Learning rate schedules

Why not just pick one fixed lr: a fixed learning rate is a compromise. Large enough to make fast progress early risks overshooting/oscillating once the loss gets close to its minimum; small enough for a precise final convergence is painfully slow early on, when big steps toward the minimum are still cheap and safe to take.

Step decay: multiply lr by a fixed factor every N epochs.
```
lr=0.1 for epochs 0-9, lr=0.1*0.5=0.05 for epochs 10-19, lr=0.025 for epochs 20-29, ...
```
Cosine annealing: smoothly decay lr following a cosine curve from an initial value down to (near) zero over the course of training, no abrupt jumps like step decay, the rate of decrease itself slows down near the end (cosine's shape flattens near 0), spending more time at small, precise steps right when it matters most.

Warmup: start with a SMALL lr and ramp UP for the first few steps, before switching to a decay schedule. Why: at initialization, weights are random (even with good init, see section 0.5), the very first gradients can be large and somewhat arbitrary, a big lr on these very first steps can push the model into a bad region it struggles to escape. Especially important for transformers (see `llm-mechanics/llm-architecture.ipynb`), which are known to be unstable to train without a warmup phase.

In [ ]:
import torch

model = torch.nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

step_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print("step decay lr over epochs:")
opt2 = torch.optim.SGD(model.parameters(), lr=0.1)
sched2 = torch.optim.lr_scheduler.StepLR(opt2, step_size=10, gamma=0.5)
for epoch in [0, 9, 10, 19, 20, 29]:
    print(f"  epoch {epoch}: lr={opt2.param_groups[0]['lr']:.4f}")
    if epoch in [9, 19, 29]:
        sched2.step()

print("\ncosine annealing lr over epochs:")
opt3 = torch.optim.SGD(model.parameters(), lr=0.1)
sched3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=30)
for epoch in range(30):
    if epoch in [0, 10, 20, 29]:
        print(f"  epoch {epoch}: lr={opt3.param_groups[0]['lr']:.4f}")
    sched3.step()